<a href="https://colab.research.google.com/github/lytyler/ST554_Homework7/blob/main/Homework7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lanette Tyler   
Homework 7   
ST554    
Spring 2026

# Introduction

## Purpose   
The purpose of this homework assignment is to practice fitting MLR and logistic regression models, including penalized or regularized models.

## Data   
The data used for this assignment is the [Wine Quality Data](https://https://archive.ics.uci.edu/dataset/186/wine+quality) from UCI Machine Learning Repository. The data set consists of two data files, one for red wine and one for white. As originally used, the predictor variables fixed acidity, volatile acidity, citric acid, residual sugar, chlorides, free sulfur dioxide, total sulfur dioxide, density, pH, sulphates, and alcohol were used to model the response variable quality. For our purposes, predictor variables will be used to model alcohol as a numeric response through MLR, and type of wine as a binary response through logistic regression.

# Preliminary Data Tasks

In [116]:
#import modules
import pandas as pd
import numpy as np

In [117]:
#read in, modify and combine data
red_wine_data = pd.read_csv("winequality-red.csv", sep = ";")
red_wine_data["wine_type"] = "Red"

white_wine_data = pd.read_csv("winequality-white.csv", sep = ";")
white_wine_data["wine_type"] = "White"

frames = [red_wine_data, white_wine_data]
wine_data = pd.concat(frames)
wine_data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,Red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,Red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,Red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,Red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,Red


In [118]:
#check concatenation results
print(red_wine_data.shape)
print(white_wine_data.shape)
print(wine_data.shape)

(1599, 13)
(4898, 13)
(6497, 13)


The combined data set has the the same number of columns and combined number of rows from the white wine and red wine data sets, so the concatenation reult looks good.

# Regression Task   
For the regression task, the alcohol value will be used as the response variable. Four different multiple linear regression models will be fit, along with Lasso, Ridge Regression, and Elastic Net models. First the data will be split into train and test sets, and the test set will be standardized. CV will be used on the training set to determine the best of each model type, and the final models of each type will be compared on the test set using both RMSE and MAE as model metrics.

## Split Data into Train and Test Sets, Standardize Test Set

In [119]:
#import function
from sklearn.model_selection import train_test_split

In [120]:
#split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    wine_data.drop(columns = ["alcohol", "quality"]),
    wine_data["alcohol"],
    test_size = 0.2,
    random_state = 39,
    stratify = wine_data.wine_type)

In [121]:
#check stratification
print(X_train["wine_type"].value_counts())
print(X_test["wine_type"].value_counts())

wine_type
White    3918
Red      1279
Name: count, dtype: int64
wine_type
White    980
Red      320
Name: count, dtype: int64


About 25% of the wines in both X_train and X-test are red wines, so the
stratitifcation worked.

In [122]:
#save means and stds from training set to standardize test set later
means = X_train.drop(columns = "wine_type").apply(np.mean, axis = 0)
print(means)
stds = X_train.drop(columns = "wine_type").apply(np.std, axis = 0)
print(stds)

fixed acidity             7.225476
volatile acidity          0.339466
citric acid               0.318453
residual sugar            5.432961
chlorides                 0.056127
free sulfur dioxide      30.538676
total sulfur dioxide    115.979123
density                   0.994704
pH                        3.218615
sulphates                 0.532898
dtype: float64
fixed acidity            1.300596
volatile acidity         0.165364
citric acid              0.145958
residual sugar           4.702408
chlorides                0.034970
free sulfur dioxide     17.937512
total sulfur dioxide    56.619682
density                  0.002931
pH                       0.161621
sulphates                0.149673
dtype: float64


In [123]:
#standardize training set, leaving wine_type unchanged since it isn't numeric
X_train =  X_train.apply(lambda x: ((x - np.mean(x))/np.std(x)) if isinstance(x[0], float) else x, axis = 0)
X_train.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,wine_type
2515,-0.096476,-1.327169,-0.126426,-0.496121,-1.090262,-0.866267,-1.394906,-1.591259,-1.228892,-0.086174,White
692,-0.942242,0.124174,-1.085606,0.184382,-0.575537,0.304464,1.324996,-0.172045,-0.300794,0.581951,White
4635,-1.096018,0.033465,-1.154119,1.141338,-0.661324,-0.253027,0.071016,0.080412,0.194192,-0.286611,White
1215,1.210617,-0.420080,0.969779,-0.708778,1.111616,-0.587522,-1.536199,0.059942,0.256065,0.181076,Red
3729,0.211075,-0.359607,0.627215,-0.878903,-0.346770,-1.089263,0.176986,-0.605314,0.379811,-0.553861,White


In [124]:
#check standardization, means should be 0 and stds should be 1
print(X_train.drop(columns = "wine_type").apply(np.mean, axis = 0))
print(X_train.drop(columns = "wine_type").apply(np.std, axis = 0))

fixed acidity          -7.123201e-16
volatile acidity        1.585972e-16
citric acid             1.897014e-16
residual sugar          1.490267e-16
chlorides              -3.862388e-17
free sulfur dioxide    -6.015755e-17
total sulfur dioxide    7.109529e-17
density                -5.373573e-14
pH                     -1.100610e-16
sulphates              -4.839949e-16
dtype: float64
fixed acidity           1.0
volatile acidity        1.0
citric acid             1.0
residual sugar          1.0
chlorides               1.0
free sulfur dioxide     1.0
total sulfur dioxide    1.0
density                 1.0
pH                      1.0
sulphates               1.0
dtype: float64


## Create MLR Models and Select Best

In [125]:
#import functions
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

In [126]:
#create MLR models, and view associated errors

#all numeric predictors
MLR1 = cross_validate(
    LinearRegression(),
    X_train[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates']],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error"
)

#scaled back numeric predictors
MLR2 = cross_validate(
    LinearRegression(),
    X_train[['fixed acidity', 'residual sugar', 'pH', 'sulphates']],
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error"
)

#with interaction, first use Polynomial Features to create design
poly3 = PolynomialFeatures(interaction_only = True, include_bias = False)
MLR3 = cross_validate(
    LinearRegression(),
    poly3.fit_transform(X_train[["fixed acidity", "residual sugar",
                                 'pH', 'sulphates']]),
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error"
)

#with polynomial, first use Polynomial Features to create design
poly4 = PolynomialFeatures()
MLR4 = cross_validate(
    LinearRegression(),
    poly4.fit_transform(X_train[["fixed acidity", "residual sugar",
                                 'pH', 'sulphates']]),
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error"
)


print("RMSE for MLR1: ", np.sqrt(-sum(MLR1['test_score'])))
print("RMSE for MLR2: ", np.sqrt(-sum(MLR2['test_score'])))
print("RMSE for MLR3: ", np.sqrt(-sum(MLR3['test_score'])))
print("RMSE for MLR4: ", np.sqrt(-sum(MLR4['test_score'])))

RMSE for MLR1:  1.1548428714922143
RMSE for MLR2:  2.4450964169567886
RMSE for MLR3:  2.4137447809822676
RMSE for MLR4:  2.400479047720422


The first MLR model (MLR1) shows the least error. This model included all numeric predictors, but no interaction or polynomial terms.

In [127]:
#fit best MLR model on whole training set, and view intercept and coefficients
best_MLR = LinearRegression().fit(X_train[['fixed acidity', 'volatile acidity',
                                           'citric acid', 'residual sugar',
                                           'chlorides', 'free sulfur dioxide',
                                           'total sulfur dioxide', 'density',
                                            'pH', 'sulphates']],
                                  y_train)
print("Best MLR Intercept: ", best_MLR.intercept_)
print("\nBest MLR Coefficients:\n",
      np.array(list(zip(X_train.columns, best_MLR.coef_))))

Best MLR Intercept:  10.486563401962572

Best MLR Coefficients:
 [['fixed acidity' '0.7722280240809553']
 ['volatile acidity' '0.21971004629758661']
 ['citric acid' '0.0488520171480982']
 ['residual sugar' '0.9695811514596372']
 ['chlorides' '0.012543202758867666']
 ['free sulfur dioxide' '0.02396597902161396']
 ['total sulfur dioxide' '-0.2429533085969239']
 ['density' '-1.8354150537719958']
 ['pH' '0.4856070432426265']
 ['sulphates' '0.22384896003235602']]


The greatest magnitude coefficients (for fitting the scaled data set) are for density, residual sugar, fixed acidity, and pH. The coefficients for citric acid, chlorides, and free sulfur dioxide are very small.

## Fit LASSO Model and Select Tuning Parameter/Best Fit

In [128]:
#import function
from sklearn.linear_model import LassoCV, Lasso

In [129]:
#select tuning parameter using CV, extract model coefficients fit on entire training set
lasso_model = LassoCV(cv = 5, random_state = 33) \
    .fit(X_train[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates']], y_train) #include all numeric predictors
print("Best Lasso Model Alpha: ", lasso_model.alpha_) #best fit tuning parameter
print("\nBest LASSO Model Coefficients:\n",
      np.array(list(zip(X_train.columns, lasso_model.coef_)))) #corresponding coefficients

Best Lasso Model Alpha:  0.0008330265651828055

Best LASSO Model Coefficients:
 [['fixed acidity' '0.765938726424469']
 ['volatile acidity' '0.21749148032801602']
 ['citric acid' '0.04813535901069731']
 ['residual sugar' '0.9599157807452655']
 ['chlorides' '0.009915546703016965']
 ['free sulfur dioxide' '0.02131018318706444']
 ['total sulfur dioxide' '-0.2406357710080959']
 ['density' '-1.8244934424753287']
 ['pH' '0.4812560396394119']
 ['sulphates' '0.22272590448860252']]


The optimal alpha for the LASSO model with all numeric predictors is close to zero, 0.000833. As would be expected as alpha approaches zero, the coefficients are very similar to the coefficients for the MLR model with the same predictors. These predictors seem to be  "going away": citric acid, chlorides, and free sulfur dioxide, while density, residual sugar, fixed acidity, and pH seem to dominate.

## Fit Ridge Regression Model and Select Tuning Parameter/Best Fit

In [130]:
#import function
from sklearn.linear_model import RidgeCV, Ridge

In [131]:
#select tuning parameter using CV, extract model coefficients fit on entire training set
ridgeR_model = RidgeCV(alphas = [0.01, 0.1, 1.0, 10.0, 100.0], cv = 5) \
    .fit(X_train[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates']], y_train) #include all numeric predictors)
print("Best Ridge Regression Model Tuning Parameter: ", ridgeR_model.alpha_)
print("\nBest Ridge Regression Model Coefficients:\n",
      np.array(list(zip(X_train.columns, ridgeR_model.coef_))))

Best Ridge Regression Model Tuning Parameter:  1.0

Best Ridge Regression Model Coefficients:
 [['fixed acidity' '0.7704880852948547']
 ['volatile acidity' '0.21942859946285212']
 ['citric acid' '0.04908446160612116']
 ['residual sugar' '0.9672581248274953']
 ['chlorides' '0.011957096100126092']
 ['free sulfur dioxide' '0.023959982329983007']
 ['total sulfur dioxide' '-0.24302025613320935']
 ['density' '-1.8326652992591836']
 ['pH' '0.48461322993159667']
 ['sulphates' '0.22361279390252262']]


The best model fit is at tuning parameter alpha = 1.0. The corresponding coefficients are very similar to those for the best MLR and Lasso models, but vary from them a bit more than they vary from each other.

## Fit Elastic Net Model and Select Tuning Paremeters/Best Fit

In [132]:
#import function
from sklearn.linear_model import ElasticNetCV, ElasticNet

In [133]:
#select tuning paramaters with cross validation and fit best model
ElasticNet_model = ElasticNetCV(cv = 5, random_state = 15,
                                l1_ratio = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95,
                                            0.96, 0.97, 0.98, 0.99, 1.0],
                                n_alphas = 50) \
    .fit(X_train[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates']], y_train) #include all numeric predictors
print("Best Elastic Net Model Alpha: ", ElasticNet_model.alpha_)
print("\nBest Elastic Net Model L1 Ratio: ", ElasticNet_model.l1_ratio_)
print("\nBest Elastic Net Model Coefficients:\n",
      np.array(list(zip(X_train.columns, ElasticNet_model.coef_))))

Best Elastic Net Model Alpha:  0.0008330265651828055

Best Elastic Net Model L1 Ratio:  1.0

Best Elastic Net Model Coefficients:
 [['fixed acidity' '0.765938726424469']
 ['volatile acidity' '0.21749148032801602']
 ['citric acid' '0.04813535901069731']
 ['residual sugar' '0.9599157807452655']
 ['chlorides' '0.009915546703016965']
 ['free sulfur dioxide' '0.02131018318706444']
 ['total sulfur dioxide' '-0.2406357710080959']
 ['density' '-1.8244934424753287']
 ['pH' '0.4812560396394119']
 ['sulphates' '0.22272590448860252']]


L1 ratio is 1, which means this elastic net model is basically a Lasso model, and the coefficients correspond very closely to the best Lasso model fit above. They are consistently a tad smaller.

## Compare the Linear Regression Models on the Test Set

In [134]:
#standardize the test set the same way the training set was done
X_test_reg_comp = X_test.drop(columns = "wine_type")

for x in X_test_reg_comp.columns:
    X_test_reg_comp[x] = (X_test_reg_comp[x] - means[x])/stds[x]
X_test_reg_comp.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates
267,0.518627,0.063702,0.969779,-0.389792,0.625487,-0.866267,-1.394906,0.885543,0.812924,2.185450
2672,-0.173364,-1.145751,-0.126426,1.672981,-0.203791,0.861955,0.071016,0.796842,-0.115174,-1.155173
1608,-0.250252,-0.420080,1.175318,3.842083,0.024975,1.586693,2.102111,2.625446,-1.476385,-0.420236
2302,-0.942242,-0.480552,-0.948581,0.333242,-0.032216,1.084951,1.377982,0.411335,-0.053301,-0.687486
551,1.518168,-0.541025,0.147625,-0.900169,-0.861495,0.025718,-0.405850,-1.059054,-1.785751,-1.088360


In [135]:
#import functions for model metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [136]:
#fit best of each model class on test set
MLRpreds = best_MLR.predict(X_test_reg_comp)
LASSOpreds = lasso_model.predict(X_test_reg_comp)
RIDGEpreds = ridgeR_model.predict(X_test_reg_comp)
ENpreds = ElasticNet_model.predict(X_test_reg_comp)

In [137]:
#compute and compare RMSE's
print("MLR RMSE: ", np.sqrt(mean_squared_error(y_test, MLRpreds)))
print("LASSO RMSE: ", np.sqrt(mean_squared_error(y_test, LASSOpreds)))
print("Ridge RMSE: ", np.sqrt(mean_squared_error(y_test, RIDGEpreds)))
print("Elastic Net RMSE: ", np.sqrt(mean_squared_error(y_test, ENpreds)))

MLR RMSE:  0.6636944565375165
LASSO RMSE:  0.6632476684127492
Ridge RMSE:  0.66353473455289
Elastic Net RMSE:  0.6632476684127492


The LASSO model and the Elastic Net model share the same RMSE, which is the lowest. This makes sense since the best fit Elastic Net model had alpha = 1,  collapsing to a LASSO model. The RMSE's were all very close, indicating that the models perform similarly, probably due to having the same predictors.

In [138]:
#compute and compare MAE's
print("MLR MAE: ", mean_absolute_error(y_test, MLRpreds))
print("LASSO MAE: ", mean_absolute_error(y_test, LASSOpreds))
print("Ridge MAE: ", mean_absolute_error(y_test, RIDGEpreds))
print("Elastic Net MAE: ", mean_absolute_error(y_test, ENpreds))

MLR MAE:  0.4000026522865745
LASSO MAE:  0.4001230535453631
Ridge MAE:  0.40004384745093635
Elastic Net MAE:  0.4001230535453631


The models also performed similarly according to the MAE metric. In this case, the MLR best-fit model (the full MLR model) performed slightly better than the others. Once again, the errors for the LASSO and Elastic NEts models are the same due to alpha = 1 for the Elastic Net model collapsing it to a LASSO model.

# Classification Task    
For the classification task, the wine_type (red or white) will be used as the response variable. Four different logistic regression models will be fit, along with Lasso, Ridge Regression, and Elastic Net models. First the data will be split into train and test sets, and the test set will be standardized. Log loss will be used during the training process to determine the best of each model type, and the final models of each type will be compared on the test set using both accuracy and log loss as model metrics.



## Split Data into Train and Test Sets and Standardize Training Sets

In [139]:
#split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    wine_data.drop(columns = ["wine_type"]),
    wine_data["wine_type"],
    test_size = 0.2,
    random_state = 39,
    stratify = wine_data.wine_type)

In [140]:
#check stratification
print(y_train.value_counts())
print(y_test.value_counts())

wine_type
White    3918
Red      1279
Name: count, dtype: int64
wine_type
White    980
Red      320
Name: count, dtype: int64


About 25% of the wines in both y_train and y-test are red wines, so the
stratitifcation worked.

In [141]:
##save means and stds from training set to standardize test set later
means = X_train.apply(np.mean, axis = 0)
print(means)
stds = X_train.apply(np.std, axis = 0)
print(stds)

fixed acidity             7.225476
volatile acidity          0.339466
citric acid               0.318453
residual sugar            5.432961
chlorides                 0.056127
free sulfur dioxide      30.538676
total sulfur dioxide    115.979123
density                   0.994704
pH                        3.218615
sulphates                 0.532898
alcohol                  10.486563
quality                   5.815855
dtype: float64
fixed acidity            1.300596
volatile acidity         0.165364
citric acid              0.145958
residual sugar           4.702408
chlorides                0.034970
free sulfur dioxide     17.937512
total sulfur dioxide    56.619682
density                  0.002931
pH                       0.161621
sulphates                0.149673
alcohol                  1.188642
quality                  0.873219
dtype: float64


In [142]:
#standardize training set
X_train =  X_train.apply(lambda x: ((x - np.mean(x))/np.std(x)), axis = 0)
X_train.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
2515,-0.096476,-1.327169,-0.126426,-0.496121,-1.090262,-0.866267,-1.394906,-1.591259,-1.228892,-0.086174,1.189119,1.356068
692,-0.942242,0.124174,-1.085606,0.184382,-0.575537,0.304464,1.324996,-0.172045,-0.300794,0.581951,-0.577603,-0.934308
4635,-1.096018,0.033465,-1.154119,1.141338,-0.661324,-0.253027,0.071016,0.080412,0.194192,-0.286611,-0.409344,0.210880
1215,1.210617,-0.420080,0.969779,-0.708778,1.111616,-0.587522,-1.536199,0.059942,0.256065,0.181076,0.684341,0.210880
3729,0.211075,-0.359607,0.627215,-0.878903,-0.346770,-1.089263,0.176986,-0.605314,0.379811,-0.553861,-0.156955,-0.934308


## Create Logistic Regression Models and Select Best

In [143]:
#import function
from sklearn.linear_model import LogisticRegression

In [144]:
#all numeric predictors
LogR1 = cross_validate(
    LogisticRegression(penalty = None),
    X_train,
    y_train,
    cv = 5,
    scoring = "neg_log_loss")

#scaled back numeric predictors
LogR2 = cross_validate(
    LogisticRegression(penalty = None),
    X_train[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'alcohol']],
    y_train,
    cv = 5,
    scoring = "neg_log_loss"
)

#with interaction, first use Polynomial Features to create design
poly3 = PolynomialFeatures(interaction_only = True, include_bias = False)
LogR3 = cross_validate(
    LogisticRegression(penalty = None),
    poly3.fit_transform(X_train[["fixed acidity", "volatile acidity", "citric acid",
                                 "residual sugar","pH", "alcohol"]]),
    y_train,
    cv = 5,
    scoring = "neg_log_loss"
)

#with polynomial, first use Polynomial Features to create design
poly4 = PolynomialFeatures()
LogR4 = cross_validate(
    LogisticRegression(penalty = None),
    poly4.fit_transform(X_train[["fixed acidity", "residual sugar",
                                 "pH", "alcohol"]]),
    y_train,
    cv = 5,
    scoring = "neg_log_loss"
)


#compare models by averaging fold test scores
print("Mean Log Loss for LogR1: ", -(LogR1["test_score"].mean()))
print("Mean Log Loss for LogR2: ", -(LogR2["test_score"].mean()))
print("Mean Log Loss for LogR3: ", -(LogR3["test_score"].mean()))
print("Mean Log Loss for LogR4: ", -(LogR4["test_score"].mean()))

Mean Log Loss for LogR1:  0.029244382291745474
Mean Log Loss for LogR2:  0.19050504034072657
Mean Log Loss for LogR3:  0.12545895042674735
Mean Log Loss for LogR4:  0.2225414631364994


The best fit, lowest mean log loss logistic regression model is the first one, LogR1. It is the full model with all numeric predictors but without interaction or polynomial terms.

In [145]:
#fit best logistic regression model on whole training set, and view intercept
#and coefficients
best_LogR = LogisticRegression(penalty = None).fit(X_train, y_train)
print("Best Logistic Regression Model Intercept: ", best_LogR.intercept_)
print("\nBest Logistic Regression Model Coefficients:\n",
      np.array(list(zip(best_LogR.feature_names_in_, best_LogR.coef_[0]))))

Best Logistic Regression Model Intercept:  [3.87382335]

Best Logistic Regression Model Coefficients:
 [['fixed acidity' '1.279495476959487']
 ['volatile acidity' '-1.2082593735660359']
 ['citric acid' '0.2940683013528826']
 ['residual sugar' '4.661567284324368']
 ['chlorides' '-0.7055887465612831']
 ['free sulfur dioxide' '-0.9844607405741064']
 ['total sulfur dioxide' '3.156624917339922']
 ['density' '-7.276285198311522']
 ['pH' '0.7063547636047528']
 ['sulphates' '-0.2485891015422682']
 ['alcohol' '-2.7395710363843904']
 ['quality' '-0.713923863831996']]


## Fit LASSO/L1 Penalty Model and Select Best

In [146]:
#import function
from sklearn.linear_model import LogisticRegressionCV

In [147]:
#select tuning parameter using CV, extract best-fit coefficients
logistic_LASSO_model = LogisticRegressionCV(cv = 5, penalty = "l1", scoring = "neg_log_loss",
                                      solver = "liblinear", random_state = 11, Cs = 250) \
    .fit(X_train, y_train)

print("Logistic Regression L1 Penalty Model C value: ", logistic_LASSO_model.C_)
print("\nLogistic Regression L1 Penalty Model Best_Fit Coefficients:")
print(np.array(list(zip(logistic_LASSO_model.feature_names_in_,
                        logistic_LASSO_model.coef_[0]))))

Logistic Regression L1 Penalty Model C value:  [1.5021276]

Logistic Regression L1 Penalty Model Best_Fit Coefficients:
[['fixed acidity' '0.9022714311403429']
 ['volatile acidity' '-1.1928902160458452']
 ['citric acid' '0.26275873343793676']
 ['residual sugar' '4.03144710606939']
 ['chlorides' '-0.7116347313334606']
 ['free sulfur dioxide' '-0.7728003167159843']
 ['total sulfur dioxide' '2.96928054628403']
 ['density' '-6.386531648733053']
 ['pH' '0.4151952013842228']
 ['sulphates' '-0.28326100609399557']
 ['alcohol' '-2.309034654152291']
 ['quality' '-0.6569711875914966']]


The coefficients of the best-fit logistic regression LASSO model generally show shrinkage in comparison to the best fit non-regularized logistic regression model above, but there appears to be no variable selection.

## Fit L2 Penalty/Ridge Model and Select Best

In [148]:
#select tuning parameter with cross validation, and ecxtract best-fit model coefficients
logistic_Ridge_model = LogisticRegressionCV(cv = 5, solver = "newton-cg",
                                            penalty = "l2", Cs = 250,
                                            scoring = "neg_log_loss", random_state = 11) \
    .fit(X_train, y_train)

print("Logistic Regression L2 Penalty Model C value: ", logistic_Ridge_model.C_)
print("\nLogistic Regression L2 Penalty Model Best-Fit Coefficients:")
print(np.array(list(zip(logistic_Ridge_model.feature_names_in_,
                        logistic_Ridge_model.coef_[0]))))

Logistic Regression L2 Penalty Model C value:  [9.5481598]

Logistic Regression L2 Penalty Model Best-Fit Coefficients:
[['fixed acidity' '1.0604209370156208']
 ['volatile acidity' '-1.2218941001749457']
 ['citric acid' '0.27727725988938995']
 ['residual sugar' '4.273889750112641']
 ['chlorides' '-0.7204689733967091']
 ['free sulfur dioxide' '-0.8780752787155073']
 ['total sulfur dioxide' '3.073749694699978']
 ['density' '-6.723311324144712']
 ['pH' '0.5440854256081527']
 ['sulphates' '-0.2902252383057415']
 ['alcohol' '-2.462734042022798']
 ['quality' '-0.6993410766085133']]


The coefficients for the logistic regression Ridge model are generally shrunk in comparision to the non-penalized logistic regression model, but not as much as the logistic regression LASSO model.

## Fit Elastic Net/L1 and L2 Penalty Model and Select Best

In [149]:
#use cross validation to select tuning parameters, and extract best-fit model coefficients
logEN_model = LogisticRegressionCV(cv = 5, solver = "saga", penalty = "elasticnet",
                                   l1_ratios = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95,
                                                0.97, 0.98, 1.0],
                                   Cs = 50, scoring = "neg_log_loss",
                                   random_state = 11) \
    .fit(X_train[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
                  'pH', 'alcohol']], y_train)

print("Logistic Regression Elastic Net Model C value: ", logEN_model.C_)
print("\nLogistic Regression Elastic Net Model Best-Fit Coefficients:")
print(np.array(list(zip(logEN_model.feature_names_in_,
                        logEN_model.coef_[0]))))

Logistic Regression Elastic Net Model C value:  [0.5689866]

Logistic Regression Elastic Net Model Best-Fit Coefficients:
[['fixed acidity' '-2.6624322358452073']
 ['volatile acidity' '-2.210946600347172']
 ['citric acid' '0.11111125522075027']
 ['residual sugar' '1.8237572698923474']
 ['pH' '-1.8605075405001603']
 ['alcohol' '0.00680996990535086']]


The coefficients vary quite a bit in sign and magnitude from the L1 and L2 penalized logistic regression models, presumably due to the different selection of variables fit.


## Compare the Logistic Regression Models on the Test Set

In [150]:
#standardize the test set like the training set was standardized
for x in X_test.columns:
    X_test[x] = (X_test[x] - means[x])/stds[x]
X_test.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
267,0.518627,0.063702,0.969779,-0.389792,0.625487,-0.866267,-1.394906,0.885543,0.812924,2.185450,1.946285,2.501256
2672,-0.173364,-1.145751,-0.126426,1.672981,-0.203791,0.861955,0.071016,0.796842,-0.115174,-1.155173,-0.829992,1.356068
1608,-0.250252,-0.420080,1.175318,3.842083,0.024975,1.586693,2.102111,2.625446,-1.476385,-0.420236,-1.587159,-0.934308
2302,-0.942242,-0.480552,-0.948581,0.333242,-0.032216,1.084951,1.377982,0.411335,-0.053301,-0.687486,-1.250640,-0.934308
551,1.518168,-0.541025,0.147625,-0.900169,-0.861495,0.025718,-0.405850,-1.059054,-1.785751,-1.088360,0.684341,1.356068


In [151]:
#import functions for model metrics
from sklearn.metrics import accuracy_score, log_loss

In [152]:
#fit best model of each class on test set
LogR_preds = best_LogR.predict(X_test)
logistic_LASSO_model_preds = logistic_LASSO_model.predict(X_test)
logistic_Ridge_model_preds = logistic_Ridge_model.predict(X_test)
logEN_model_preds = logEN_model.predict(X_test[['fixed acidity',
                                                'volatile acidity', 'citric acid',
                                                'residual sugar', 'pH', 'alcohol']])

In [153]:
#compute and compare accuracy scores
print("Unpenalized Logistic Regression Accuracy Score: ", accuracy_score(y_test,
                                                             LogR_preds))
print("Logistic Regression LASSO Accuracy Score: ", accuracy_score(y_test,
                                                        logistic_LASSO_model_preds))
print("Logistic Regression Ridge Accuracy Score: ", accuracy_score(y_test,
                                                        logistic_Ridge_model_preds))
print("Logistic Regression Elastic Net Accuracy Score: ", accuracy_score(y_test, logEN_model_preds))

Unpenalized Logistic Regression Accuracy Score:  0.9915384615384616
Logistic Regression LASSO Accuracy Score:  0.9915384615384616
Logistic Regression Ridge Accuracy Score:  0.9915384615384616
Logistic Regression Elastic Net Accuracy Score:  0.9492307692307692


The best-fit accuracy scores for unpenalized logistic regression model, logistic regression LASSO model, and logistic regression ridge model were all the same at 0.9915, and higher than the accuracy of the logistic regression elastic net model at 0.9492. The elastic net model was fit on a smaller range of variables, while the other three were all fit on all variables, contributing to the high accuracy score.

In [154]:
#get probability estimates for class predictions
LogR_probs = best_LogR.predict_proba(X_test)
logistic_LASSO_model_probs = logistic_LASSO_model.predict_proba(X_test)
logistic_Ridge_model_probs = logistic_Ridge_model.predict_proba(X_test)
logEN_model_probs = logEN_model.predict_proba(X_test[['fixed acidity',
                                                'volatile acidity', 'citric acid',
                                                'residual sugar', 'pH', 'alcohol']])

In [155]:
#compute and compare log loss metric
print("Unpenalized Logistic Regression Log Loss: ", log_loss(y_test,
                                                             LogR_probs))
print("Logistic Regression LASSO Log Loss: ", log_loss(y_test,
                                                        logistic_LASSO_model_probs))
print("Logistic Regression Ridge Log Loss: ", log_loss(y_test,
                                                        logistic_Ridge_model_probs))
print("Logistic Regression Elastic Net Log Loss: ", log_loss(y_test, logEN_model_probs))

Unpenalized Logistic Regression Log Loss:  0.062167569859597094
Logistic Regression LASSO Log Loss:  0.06276672731351098
Logistic Regression Ridge Log Loss:  0.0625358647445788
Logistic Regression Elastic Net Log Loss:  0.1270803743450068


The log loss values for unpenalized (0.0622), LASSO (0.0628), and ridge (0.0625) logistic regression models are very similar, although not identical. They are lower, and thus better, than the 0.1271 log loss for the elastic net model, which had fewer variables fit.